In [0]:
# ==========================================
# 1. LOAD ENGINEERED DATA
# ==========================================

df = spark.table("CCC_FEATURES")

display(df.limit(5))

# Convert Spark DataFrame to Pandas
pdf = df.toPandas()

print("Dataset shape:", pdf.shape)


# ==========================================
# 2. HANDLE PAYMENT-TO-BALANCE MISSING VALUES
# ==========================================

# PAYMENT_TO_BALANCE is undefined when the customer
# does not have a positive average bill balance.
# Create an indicator to preserve this information.

pdf["NO_POSITIVE_BALANCE"] = (
    pdf["AVG_BILL_AMT"] <= 0
).astype(int)

# Fill the undefined payment-to-balance ratio with 0.
# The indicator above allows the model to distinguish these
# customers from customers with a genuine ratio of 0.

pdf["PAYMENT_TO_BALANCE"] = (
    pdf["PAYMENT_TO_BALANCE"].fillna(0)
)


# ==========================================
# 3. DEFINE TARGET AND PREDICTORS
# ==========================================

features = [

    "LIMIT_BAL",

    "SEX",

    "EDUCATION",

    "MARRIAGE",

    "AGE",

    "PAY_1",

    "PAY_2",

    "PAY_3",

    "PAY_4",

    "PAY_5",

    "PAY_6",

    "BILL_AMT1",

    "BILL_AMT2",

    "BILL_AMT3",

    "BILL_AMT4",

    "BILL_AMT5",

    "BILL_AMT6",

    "PAY_AMT1",

    "PAY_AMT2",

    "PAY_AMT3",

    "PAY_AMT4",

    "PAY_AMT5",

    "PAY_AMT6",

    "AVG_BILL_AMT",

    "AVG_PAY_AMT",

    "AVG_UTILIZATION",

    "PAYMENT_TO_BALANCE",

    "NO_POSITIVE_BALANCE",

    "DELINQUENT_MONTHS",

    "MAX_DELINQUENCY",

    "BALANCE_CHANGE"

]

X = pdf[features]

y = pdf["dpnm"]

print("Predictor shape:", X.shape)
print("Target shape:", y.shape)


# ==========================================
# 4. TRAIN/TEST SPLIT
# ==========================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.20,

    random_state=42,

    stratify=y

)

print("Training observations:", len(X_train))
print("Testing observations:", len(X_test))

print(
    "Training default rate:",
    round(y_train.mean(), 4)
)

print(
    "Testing default rate:",
    round(y_test.mean(), 4)
)


# ==========================================
# 5. FINAL DATA QUALITY CHECK
# ==========================================

import numpy as np

print(
    "Training missing values:",
    X_train.isna().sum().sum()
)

print(
    "Testing missing values:",
    X_test.isna().sum().sum()
)

print(
    "Training infinite values:",
    np.isinf(X_train).sum().sum()
)

print(
    "Testing infinite values:",
    np.isinf(X_test).sum().sum()
)



# ==========================================
# 6. STANDARDIZE FEATURES
# ==========================================

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ==========================================
# 7. TRAIN LOGISTIC REGRESSION
# ==========================================

from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

log_model.fit(X_train_scaled, y_train)

y_pred = log_model.predict(X_test_scaled)

y_prob = log_model.predict_proba(X_test_scaled)[:, 1]

# ==========================================
# 8. EVALUATE LOGISTIC REGRESSION
# ==========================================

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print("Logistic Regression Results")
print("---------------------------")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")


#Confusion matrix
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

tn, fp, fn, tp = cm.ravel()

print("\nTrue Negatives:", tn)
print("False Positives:", fp)
print("False Negatives:", fn)
print("True Positives:", tp)


## Baseline Model Interpretation

#The logistic regression model achieved an ROC-AUC of 0.7433 and overall
#accuracy of 80.35%. However, because only 22.12% of customers defaulted,
#accuracy alone is not an appropriate measure of model effectiveness.

#At the standard 0.50 classification threshold, the model identified 390 of
#1,327 customers who ultimately defaulted, resulting in recall of 29.39%.
#Although precision was relatively strong at 61.71%, the model failed to
#identify 937 eventual defaulters.

#These results establish a baseline for evaluating whether a nonlinear
#gradient-boosting model can better identify customers at risk of default.

ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_1,PAY_2,PAY_3,PAY_4,PAY_5,PAY_6,BILL_AMT1,BILL_AMT2,BILL_AMT3,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,dpnm,AVG_BILL_AMT,AVG_PAY_AMT,AVG_UTILIZATION,PAYMENT_TO_BALANCE,DELINQUENT_MONTHS,MAX_DELINQUENCY,BALANCE_CHANGE
1,20000,2,2,1,24,2,2,-1,-1,-2,-2,3913,3102,689,0,0,0,0,689,0,0,0,0,1,1284.0,114.83333333333333,0.0642,0.08943406022845275,2,2,3913
2,120000,2,2,2,26,-1,2,0,0,0,2,2682,1725,2682,3272,3455,3261,0,1000,1000,1000,0,2000,1,2846.1666666666665,833.3333333333334,0.023718055555555555,0.2927914739122797,2,2,-579
3,90000,2,2,2,34,0,0,0,0,0,0,29239,14027,13559,14331,14948,15549,1518,1500,1000,1000,1000,5000,0,16942.166666666668,1836.3333333333333,0.1882462962962963,0.10838834072776996,0,0,13690
4,50000,2,2,1,37,0,0,0,0,0,0,46990,48233,49291,28314,28959,29547,2000,2019,1200,1100,1069,1000,0,38555.666666666664,1398.0,0.7711133333333333,0.03625926150068732,0,0,17443
5,50000,1,2,1,57,-1,0,-1,0,0,0,8617,5670,35835,20940,19146,19131,2000,36681,10000,9000,689,679,0,18223.166666666668,9841.5,0.36446333333333336,0.5400543264525923,0,0,-10514


Dataset shape: (30000, 32)
Predictor shape: (30000, 31)
Target shape: (30000,)
Training observations: 24000
Testing observations: 6000
Training default rate: 0.2212
Testing default rate: 0.2212
Training missing values: 0
Testing missing values: 0
Training infinite values: 0
Testing infinite values: 0
Logistic Regression Results
---------------------------
Accuracy:  0.8035
Precision: 0.6171
Recall:    0.2939
F1 Score:  0.3982
ROC-AUC:   0.7433
Confusion Matrix:
[[4431  242]
 [ 937  390]]

True Negatives: 4431
False Positives: 242
False Negatives: 937
True Positives: 390
